<a href="https://colab.research.google.com/github/ekonjmrivas-devops/llm_engineering/blob/mis-ejercicios/W3_PRACTICA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semana 3 - Práctica


**Bloque - Librerías**

In [ ]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 957.0/957.0 kB 29.1 MB/s eta 0:00:00


In [ ]:
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
#Instalar librería para PDF (si no está instalada)

!pip install -q fpdf2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 15.9 MB/s eta 0:00:00


In [ ]:
!pip install -q gradio

In [ ]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
from datetime import datetime
import anthropic
from fpdf import FPDF
import gradio as gr
import urllib.request
#from fpdf import FPDF, XPos, YPos
from fpdf.enums import XPos, YPos, WrapMode
import matplotlib
import shutil

**Bloque - Setup y conexión a Drive**

In [ ]:
# Conexión a Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#drive.flush_and_unmount()
#drive.mount('/content/drive', force_remount=True)

In [ ]:
# Rutas fijas del proyecto
BASE_PATH = '/content/drive/MyDrive/cursollms/week3'
INPUT_PATH = f'{BASE_PATH}/Inbound files'
OUTPUT_PATH = f'{BASE_PATH}/Outbound files'

In [ ]:
# Crear carpetas si no existen
os.makedirs(INPUT_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
print(f"Carpeta de entrada: {INPUT_PATH}")
print(f"Carpeta de salida: {OUTPUT_PATH}")

Carpeta de entrada: /content/drive/MyDrive/cursollms/week3/Inbound files
Carpeta de salida: /content/drive/MyDrive/cursollms/week3/Outbound files


**Bloque - Listar archivos de la carpeta de entrada**

In [ ]:
# Listar archivos disponibles en directorio de Drive

def listar_archivos_origen():
    """
    Lista los archivos disponibles en la carpeta de entrada de Drive.
    Devuelve una lista de nombres de archivo (sin la ruta completa),
    pensada para poblar el Dropdown de Gradio.
    """
    extensiones_validas = ('.mp3', '.wav', '.m4a', '.txt')

    archivos = [
        f for f in os.listdir(INPUT_PATH)
        if f.lower().endswith(extensiones_validas)
    ]

    archivos.sort()
    return archivos

In [ ]:
# Comprobación de la función
archivos_disponibles = listar_archivos_origen()
print(f"Archivos encontrados: {len(archivos_disponibles)}")
for a in archivos_disponibles:
    print(f"  - {a}")

Archivos encontrados: 2
  - Desarrollo de un caso práctico.txt
  - denver_extract.mp3


In [ ]:
# Diagnóstico: ver TODO lo que hay en la carpeta, sin filtrar por extensión
print(os.listdir(INPUT_PATH))

['Irregular Verbs.pdf', 'training-5-azure-devops-pipelines.md', 'Desarrollo de un caso práctico.txt', 'denver_extract.mp3']


**Bloque - Transcripción de audio con Whisper (OpenAI API)**

In [ ]:
# Crear cliente de OpenAI
client = OpenAI(api_key= userdata.get('OPENAI_API_KEY'))
AUDIO_MODEL = "whisper-1"

In [ ]:
def transcribir_audio(nombre_archivo):
    """
    Transcribe un archivo de audio a texto usando Whisper (OpenAI API).
    nombre_archivo: nombre del archivo dentro de INPUT_PATH (ej. 'denver_extract.mp3')
    Devuelve: el texto transcrito.
    """
    ruta_completa = os.path.join(INPUT_PATH, nombre_archivo)

    with open(ruta_completa, "rb") as audio_file:
        transcripcion = client.audio.transcriptions.create(
            model=AUDIO_MODEL,
            file=audio_file,
            response_format="text"
        )

    return transcripcion

In [ ]:
#texto_transcrito = transcribir_audio("denver_extract.mp3")
#print(texto_transcrito[:500])
#print(f"\nLongitud total: {len(texto_transcrito)} caracteres")

**Bloque - Lectura de archivo de texto**

In [ ]:
def leer_texto(nombre_archivo):
    """
    Lee directamente un archivo de texto (.txt) de la carpeta de entrada.
    nombre_archivo: nombre del archivo dentro de INPUT_PATH (ej. 'Desarrollo de un caso práctico.txt')
    Devuelve: el contenido del archivo como string.
    """
    ruta_completa = os.path.join(INPUT_PATH, nombre_archivo)

    with open(ruta_completa, "r", encoding="utf-8") as f:
        contenido = f.read()

    return contenido

In [ ]:
#texto_leido = leer_texto("Desarrollo de un caso práctico.txt")
#print(texto_leido[:500])
#print(f"\nLongitud total: {len(texto_leido)} caracteres")

**Bloque - Seleccionar tipo de archivo de entrada**

In [ ]:
def obtener_texto_origen(nombre_archivo, tipo_origen):
    """
    Obtiene el texto de origen, ya sea transcribiendo audio o leyendo texto directo.
    tipo_origen: "audio" o "texto"
    """
    if tipo_origen == "audio":
        return transcribir_audio(nombre_archivo)
    elif tipo_origen == "texto":
        return leer_texto(nombre_archivo)
    else:
        raise ValueError(f"Tipo de origen no reconocido: {tipo_origen}")

In [ ]:
#texto_prueba = obtener_texto_origen("Desarrollo de un caso práctico.txt", "texto")
#print(texto_prueba[:300])

**Bloque - Configuración de cuantización y carga de Llama local**

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
# Carga el modelo de Llama solo cuando sea necesario

LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

llama_tokenizer = None
llama_model = None

def cargar_llama_si_necesario():
    """
    Carga el modelo Llama solo si no está ya cargado en memoria.
    Evita recargarlo innecesariamente entre llamadas.
    """
    global llama_tokenizer, llama_model

    if llama_model is None:
        print("Cargando Llama por primera vez...")
        llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA)
        llama_tokenizer.pad_token = llama_tokenizer.eos_token
        llama_model = AutoModelForCausalLM.from_pretrained(
            LLAMA, device_map="auto", quantization_config=quant_config
        )
        print("Llama cargado correctamente.")
    else:
        print("Llama ya estaba cargado en memoria.")

**Bloque - Generación de resumen y acciones**

In [ ]:
# Configuración Claude

CLAUDE = "claude-sonnet-4-6"

claude_client = anthropic.Anthropic(
    api_key=userdata.get('ANTHROPIC_API_KEY')
)

In [ ]:
# Generar resumen con Claude

def generar_con_claude(texto):
    """
    Genera resumen y acciones usando Claude API.
    """
    messages = [
        {
            "role": "user",
            "content": f"""Eres un asistente experto en redacción de actas de reuniones.

Dado el siguiente texto de una reunión, genera:
1. Un resumen ejecutivo del acta (máximo 300 palabras)
2. Lista de acciones acordadas o próximos pasos

Texto de la reunión:
{texto}

Responde en español, con este formato exacto:
## Resumen ejecutivo
[resumen aquí]

## Acciones acordadas
- [acción 1]
- [acción 2]
..."""
        }
    ]

    response = claude_client.messages.create(
        model=CLAUDE,
        max_tokens=1000,
        messages=messages
    )

    return response.content[0].text


In [ ]:
# Generar resumen con Llama

def generar_con_llama(texto):
    """
    Genera resumen y acciones usando Llama local.
    """
    cargar_llama_si_necesario()

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Eres un asistente experto en redacción de actas de reuniones. Responde siempre en español.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Dado el siguiente texto de una reunión, genera:
1. Un resumen ejecutivo del acta (máximo 300 palabras)
2. Lista de acciones acordadas o próximos pasos

Texto:
{texto}

Responde con este formato exacto:
## Resumen ejecutivo
[resumen aquí]

## Acciones acordadas
- [acción 1]
- [acción 2]<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>"""

    inputs = llama_tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=1000
    )

    return llama_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
cargar_llama_si_necesario()

Cargando Llama por primera vez...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Llama cargado correctamente.


**Bloque - Función orquestadora del modelo para generar resumen**

In [ ]:
# Función orquestadora para generar resumen en función del modelo

def generar_resumen(texto, modelo="claude"):
    """
    Genera resumen y acciones con el modelo seleccionado.
    modelo: "claude" o "llama"
    """
    if modelo == "claude":
        return generar_con_claude(texto)
    elif modelo == "llama":
        return generar_con_llama(texto)
    else:
        raise ValueError(f"Modelo no reconocido: {modelo}")

In [ ]:
#resultado = generar_resumen(texto_prueba, modelo="claude")
#print(resultado)

NameError: name 'texto_prueba' is not defined

**Bloque - Generación automática del nombre del archivo de salida**

In [ ]:
def generar_nombre_archivo(nombre_origen, extension="txt"):
    """
    Genera un nombre de archivo de salida basado en:
    - nombre del archivo de origen (sin extensión)
    - fecha actual (YYYYMMDD)
    - hora actual (HHMMSS)
    - extensión elegida (txt o pdf)

    Ejemplo: denver_extract_20260624_143052.txt
    """
    nombre_base = os.path.splitext(nombre_origen)[0]
    ahora = datetime.now()
    timestamp = ahora.strftime("%Y%m%d_%H%M%S")

    return f"{nombre_base}_{timestamp}.{extension}"

**Bloque - Guardar resultado en Google Drive**

In [ ]:
# Guardar archivo como TXT

def guardar_txt(contenido, nombre_archivo):
    """
    Guarda el contenido en un archivo TXT en la carpeta outbound files.
    nombre_archivo: nombre con extensión (ej. 'denver_extract_20260624_143052.txt')
    Devuelve: ruta completa del archivo guardado.
    """
    ruta_completa = os.path.join(OUTPUT_PATH, nombre_archivo)

    with open(ruta_completa, "w", encoding="utf-8") as f:
        f.write(contenido)

    print(f"✅ Archivo TXT guardado en: {ruta_completa}")
    return ruta_completa

In [ ]:
# Requerida fuente para procesar caracteres especiales al generar archivo PDF

def descargar_fuente_si_necesario():
    """Obtiene la fuente DejaVu incluida en matplotlib (sin depender de internet)."""
    fuente_path = "/tmp/DejaVuSans.ttf"
    if not os.path.exists(fuente_path):
        print("Copiando fuente DejaVu desde matplotlib...")
        origen = os.path.join(matplotlib.get_data_path(), "fonts", "ttf", "DejaVuSans.ttf")
        shutil.copy(origen, fuente_path)
        print("✅ Fuente copiada")
    return fuente_path

In [ ]:
def guardar_pdf(contenido, nombre_archivo):
    ruta_completa = os.path.join(OUTPUT_PATH, nombre_archivo)
    fuente_path = descargar_fuente_si_necesario()

    pdf = FPDF()
    pdf.add_page()
    pdf.add_font("DejaVu", fname=fuente_path)
    pdf.add_font("DejaVu", style="B", fname=fuente_path)
    pdf.set_auto_page_break(auto=True, margin=15)

    import re
    contenido_limpio = re.sub(
        r'[^\x00-\x7F\u00C0-\u024F\u0400-\u04FF\n\r ]',
        '',
        contenido
    )

    for linea in contenido_limpio.split('\n'):
        if linea.startswith('## '):
            pdf.set_font("DejaVu", style="B", size=13)
            pdf.cell(
                0, 10, linea.replace('## ', ''),
                new_x=XPos.LMARGIN, new_y=YPos.NEXT
            )
        elif linea.startswith('- '):
            pdf.set_font("DejaVu", size=11)
            pdf.multi_cell(
                0, 8, f"  • {linea[2:]}",
                new_x=XPos.LMARGIN, new_y=YPos.NEXT   # ← el fix
            )
        else:
            pdf.set_font("DejaVu", size=11)
            pdf.multi_cell(
                0, 8, linea,
                new_x=XPos.LMARGIN, new_y=YPos.NEXT   # ← el fix
            )

    pdf.output(ruta_completa)
    print(f"✅ Archivo PDF guardado en: {ruta_completa}")
    return ruta_completa

**Bloque - Función orquestadora para guardar achivo**

In [ ]:
# Función orquestadora de guardado de archivo

def guardar_resultado(contenido, nombre_archivo, formato="txt"):
    """
    Guarda el resultado en el formato indicado.
    formato: "txt" o "pdf"
    Devuelve: ruta completa del archivo guardado.
    """
    if formato == "txt":
        return guardar_txt(contenido, nombre_archivo)
    elif formato == "pdf":
        return guardar_pdf(contenido, nombre_archivo)
    else:
        raise ValueError(f"Formato no reconocido: {formato}")

**Bloque - Función orquestadora principal**

In [ ]:
def procesar_reunion(nombre_archivo, tipo_origen, modelo, formato_salida, nombre_salida=None):
    """
    Orquesta el flujo completo de procesamiento de una reunión.

    Parámetros:
    - nombre_archivo: archivo seleccionado de inbound files
    - tipo_origen: "audio" o "texto"
    - modelo: "claude" o "llama"
    - formato_salida: "txt" o "pdf"
    - nombre_salida: nombre del archivo de salida (si None, se genera automáticamente)

    Devuelve: (resumen, ruta_archivo_guardado)
    """

    print(f"📂 Archivo origen: {nombre_archivo}")
    print(f"🔧 Tipo: {tipo_origen} | Modelo: {modelo} | Formato: {formato_salida}")
    print("─" * 50)

    # Paso 1 — Obtener texto (transcribir audio o leer txt)
    print("⏳ Paso 1: Obteniendo texto de origen...")
    texto = obtener_texto_origen(nombre_archivo, tipo_origen)
    print(f"✅ Texto obtenido ({len(texto)} caracteres)")

    # Paso 2 — Generar resumen y acciones
    print(f"⏳ Paso 2: Generando resumen con {modelo}...")
    resumen = generar_resumen(texto, modelo=modelo)
    print("✅ Resumen generado")

    # Paso 3 — Generar nombre de archivo si no se proporcionó
    if nombre_salida is None or nombre_salida.strip() == "":
        nombre_salida = generar_nombre_archivo(nombre_archivo, formato_salida)
    else:
        # Asegurarse de que tiene la extensión correcta
        nombre_salida = f"{os.path.splitext(nombre_salida)[0]}.{formato_salida}"

    print(f"⏳ Paso 3: Guardando resultado como {nombre_salida}...")
    ruta = guardar_resultado(resumen, nombre_salida, formato=formato_salida)
    print(f"✅ Archivo guardado en: {ruta}")
    print("─" * 50)
    print("🎉 Proceso completado")

    return resumen, ruta

#Prueba completa del flujo sin Gradio

resumen, ruta = procesar_reunion(
    nombre_archivo="Desarrollo de un caso práctico.txt",
    tipo_origen="texto",
    modelo="claude",
    formato_salida="txt"
)

print("\n--- RESUMEN GENERADO ---")
print(resumen[:500])
print(f"\n--- GUARDADO EN ---")
print(ruta)

**Bloque - Interface Gradio**

In [ ]:
def interfaz_procesar(nombre_archivo, tipo_origen, modelo, formato_salida, nombre_salida):
    """
    Función puente entre Gradio y la orquestadora principal.
    """
    try:
        resumen, ruta = procesar_reunion(
            nombre_archivo=nombre_archivo,
            tipo_origen=tipo_origen,
            modelo=modelo,
            formato_salida=formato_salida,
            nombre_salida=nombre_salida if nombre_salida.strip() != "" else None
        )
        return resumen, ruta, f"✅ Proceso completado. Archivo guardado en:\n{ruta}"
    except Exception as e:
        return "", "", f"❌ Error: {str(e)}"

In [ ]:
def refrescar_archivos():
    """Refresca la lista de archivos disponibles en inbound files."""
    return gr.Dropdown(choices=listar_archivos_origen())

In [ ]:
# Construcción de la interfaz
with gr.Blocks(title="Generador de Actas de Reuniones") as app:

    gr.Markdown("# 📝 Generador de Actas de Reuniones")
    gr.Markdown("Transcribe audio o procesa texto de reuniones y genera resumen + acciones.")

    with gr.Row():
        with gr.Column(scale=1):

            gr.Markdown("### 📂 Archivo de origen")
            archivo_input = gr.Dropdown(
                choices=listar_archivos_origen(),
                label="Selecciona archivo",
                info="Archivos disponibles en Inbound files"
            )
            btn_refrescar = gr.Button("🔄 Refrescar lista", size="sm")

            tipo_origen = gr.Radio(
                choices=["texto", "audio"],
                value="texto",
                label="Tipo de archivo",
            )

            gr.Markdown("### ⚙️ Configuración")
            modelo = gr.Radio(
                choices=["claude", "llama"],
                value="claude",
                label="Modelo de IA",
            )
            formato_salida = gr.Radio(
                choices=["txt", "pdf"],
                value="txt",
                label="Formato de salida",
            )

            gr.Markdown("### 💾 Archivo de salida")
            nombre_salida = gr.Textbox(
                label="Nombre del archivo (opcional)",
                placeholder="Si lo dejas vacío se genera automáticamente",
            )

            btn_procesar = gr.Button("🚀 Procesar", variant="primary")

        with gr.Column(scale=2):

            gr.Markdown("### 📄 Resultado")
            resumen_output = gr.Textbox(
                label="Resumen generado",
                lines=20,
                max_lines=40
            )
            ruta_output = gr.Textbox(
                label="Archivo guardado en",
                interactive=False
            )
            estado_output = gr.Textbox(
                label="Estado",
                interactive=False
            )

    # Eventos
    btn_refrescar.click(
        fn=refrescar_archivos,
        outputs=archivo_input
    )

    btn_procesar.click(
        fn=interfaz_procesar,
        inputs=[archivo_input, tipo_origen, modelo, formato_salida, nombre_salida],
        outputs=[resumen_output, ruta_output, estado_output]
    )

In [ ]:
app.launch(share=True) #, debug=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5323e87939d8abe293.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
print(descargar_fuente_si_necesario)

<function descargar_fuente_si_necesario at 0x7ec1b3cd6160>


In [ ]:
print(romper_palabras_largas)

<function romper_palabras_largas at 0x7ec1b3cd65c0>


In [ ]:
print(guardar_pdf)

In [ ]:
#librería que permite inspeccionar la función cargada en memoria
#import inspect
#print(inspect.getsource(romper_palabras_largas))

In [ ]:
!nvidia-smi

Thu Jul  2 18:09:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '69b51c1e6ed6f1c44a83db1d', 'name': 'jmrivas', 'fullname': 'Jose Maria Rivas', 'email': 'ekon.jmrivas@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1785542400, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/AKppwVi1E6KCb0gQtaArI.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'curso-llms', 'role': 'read', 'createdAt': '2026-06-02T18:38:19.662Z'}}}


Procesamiento con LLM
- Resumen del acta/minuta
- Extracción de acciones/próximos pasos

Generación del archivo de salida
- Nombre automático
- Formato TXT o PDF
- Guardado en Drive

Interfaz Gradio
- Formulario de inputs
- Preview de resultado (scroll)
- Previo del archivo de salida
